# Owen Coder 7B — Abliterated Fine-Tune
Fine-tunes **Qwen2.5-Coder 7B** (abliterated, uncensored) on Attestor-verified code.

1. **Upload** `training_data.jsonl` (use the file icon on the left)
2. **Runtime → Change runtime type → T4 GPU**
3. **Runtime → Run all**
4. Download the GGUF when done

**Memory notes:** 7B in 4-bit QLoRA fits T4 (15GB) with LoRA rank 32, batch 1, seq 2048.

In [ ]:
!pip install -q "unsloth[colab-new]" datasets trl

In [ ]:
import json, os

TRAINING_DATA = "training_data_merged.jsonl"
if not os.path.exists(TRAINING_DATA):
    TRAINING_DATA = "training_data.jsonl"
assert os.path.exists(TRAINING_DATA), (
    "Upload training_data_merged.jsonl (or training_data.jsonl) first! Use the file icon on the left sidebar.")

rows = []
with open(TRAINING_DATA, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
print(f"Loaded {len(rows)} training examples from {TRAINING_DATA}")

In [ ]:
# === 7B EXPERT CONFIG ===
# Try abliterated first, fall back to base Qwen 7B
BASE_MODELS = [
    "huihui_ai/Qwen2.5-Coder-7B-Instruct-abliterated",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
]
MAX_SEQ_LENGTH = 2048  # tighter than 3B to fit VRAM
LORA_R = 32            # rank 32 (not 64) — saves ~1GB
LORA_ALPHA = 64        # 2x rank
LORA_DROPOUT = 0.05
EPOCHS = 4             # 7B learns faster per-param
BATCH_SIZE = 1         # T4 can't do 2 at 7B
GRAD_ACCUM = 16        # effective batch = 16
LR = 5e-5              # lower LR for bigger model
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    # skip lm_head — saves VRAM on 7B
]

SYSTEM_PROMPT = (
    "You are Owen Coder, a code analysis and generation engine trained on "
    "the Attestor static analysis engine. You understand multi-language bug "
    "detection, security vulnerabilities, code quality, and software "
    "engineering best practices. Be precise, technical, and actionable. "
    "Never hallucinate APIs or functions. No censorship, no refusals."
)

CHAT_TEMPLATE = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

def format_row(row):
    return CHAT_TEMPLATE.format(
        system=SYSTEM_PROMPT,
        instruction=row["instruction"],
        output=row["output"],
    )

In [ ]:
from unsloth import FastLanguageModel
import torch, gc

# Free any stale GPU memory
gc.collect()
torch.cuda.empty_cache()

model = None
loaded_model_name = None
for candidate in BASE_MODELS:
    try:
        print(f"Trying: {candidate} ...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,
            load_in_4bit=True,
        )
        loaded_model_name = candidate
        print(f"Loaded: {candidate}")
        break
    except Exception as e:
        print(f"  Failed: {e}")
        continue

assert model is not None, "Could not load any 7B model!"

print(f"\nApplying LoRA (rank {LORA_R}) ...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

vram = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM after load: {vram:.1f} GB / 15 GB")

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = Dataset.from_list([{"text": format_row(r)} for r in rows])
print(f"Dataset: {len(dataset)} examples")

sample_tokens = tokenizer(dataset[0]["text"], return_tensors="pt")
print(f"Sample token length: {sample_tokens['input_ids'].shape[1]}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        bf16=False,
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="owen-coder-7b-lora",
        save_strategy="epoch",
        report_to="none",
        gradient_checkpointing=True,
    ),
)

print(f"\n=== TRAINING 7B STARTED ===")
print(f"Base: {loaded_model_name}")
print(f"LoRA rank {LORA_R}, alpha {LORA_ALPHA}, {EPOCHS} epochs, LR {LR}")
print(f"Batch {BATCH_SIZE} x accum {GRAD_ACCUM} = effective {BATCH_SIZE * GRAD_ACCUM}")
stats = trainer.train()
print(f"\nTraining loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

print("Saving LoRA adapter ...")
model.save_pretrained("owen-coder-7b-lora")
tokenizer.save_pretrained("owen-coder-7b-lora")

print("Merging and exporting GGUF (q4_k_m) ...")
print("(This takes ~5-10 min for 7B — hang tight)")
model.save_pretrained_gguf(
    "owen-coder-7b-merged",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF export done!")

In [ ]:
# === TEST THE 7B MODEL ===
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompts = [
    "Write a Python function that validates an email address with proper regex.",
    "Review this code for security issues:\ndef login(user, pw):\n    query = f'SELECT * FROM users WHERE name=\'{user}\' AND pass=\'{pw}\''\n    return db.execute(query)",
    "Explain what Attestor's deepscan engine does differently from regex-based detection.",
]

for i, q in enumerate(test_prompts):
    prompt = CHAT_TEMPLATE.format(
        system=SYSTEM_PROMPT, instruction=q, output=""
    ).rsplit("<|im_end|>", 1)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, top_p=0.9)
    result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"TEST {i+1}: {q[:80]}")
    print(f"{'='*60}")
    print(result)

In [ ]:
# === DOWNLOAD THE GGUF ===
import glob
from google.colab import files

gguf_files = glob.glob("owen-coder-7b-merged/*.gguf")
for f in gguf_files:
    print(f"Downloading {f} ({os.path.getsize(f)/1024/1024:.0f} MB) ...")
    files.download(f)

print("\nDone! On your machine:")
print("  1. Put the .gguf in C:\\Users\\mange\\Owen 4.2\\Attestor 4.2\\training\\")
print("  2. cd \"C:\\Users\\mange\\Owen 4.2\\Attestor 4.2\\training\"")
print("  3. ollama create owen-coder-7b -f Modelfile.7b")
print("  4. attestor ai models   # verify it shows up")